# Setup

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
len(documents)

72

# Generating Ground Truth

In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [4]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [5]:
from evaluation_utils import llm_structured_retry
import json


In [6]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [7]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "filename": doc["filename"]
        })
    
    return results, usage

In [8]:
from tqdm.auto import tqdm

In [9]:
ground_truth = []
usages =[]

for doc in tqdm(documents[:3]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

In [10]:
ground_truth

[{'question': 'What problem does retrieval-augmented generation solve for an LLM?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why does the course build the RAG system in plain Python instead of using a framework right away?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What are the main limitations of an LLM that the lesson points out?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What will the first part of the module cover before the agentic version?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What kind of app is the course building to show RAG in practice?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What do I need installed before starting this module, and do I only need Python and Jupyter?',
  'filename': '01-agentic-rag/lessons/02-environment.md'},
 {'question': 'How do I set up a fresh project with uv and which packages should I add for the lesson?',
  'fi

# Q1

In [11]:
usages

[ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=95, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1115),
 ResponseUsage(input_tokens=1286, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=123, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1409),
 ResponseUsage(input_tokens=1753, input_tokens_details=InputTokensDetails(cached_tokens=1280, cache_write_tokens=0), output_tokens=113, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1866)]

In [12]:
sum(usage.input_tokens for usage in usages) / len(usages)

1353.0

# Ground Truth and Searching Chunks

In [13]:
import pandas as pd

df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [14]:
len(ground_truth)

360

In [15]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [16]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [17]:
len(chunks)

295

In [18]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

def text_search(query, num_results=5):
    return index.search(
        query=query,
        num_results=num_results
    )

In [19]:
import sys

sys.path.append(
    "/workspaces/intro-to-RAG/02-Vector-Search/llm-zoomcamp-hw2"
)

from embedder import Embedder

embed = Embedder(
    "/workspaces/intro-to-RAG/02-Vector-Search/llm-zoomcamp-hw2/models/"
    "Xenova/all-MiniLM-L6-v2"
)

2026-07-13 22:32:41.784036102 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [20]:
texts = [chunk["content"] for chunk in chunks]

from  tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [21]:
from minsearch import VectorSearch

vindex = VectorSearch(
    keyword_fields=["filename"]
)

vindex.fit(X, chunks)


def vector_search(query, num_results=5):
    query_vector = embed.encode(query)

    return vindex.search(
        query_vector=query_vector,
        num_results=num_results
    )

In [22]:
X.shape

(295, 384)

# Q2 & Q3

In [23]:
q = ground_truth[0]["question"]

text_results = text_search(q)
vector_results = vector_search(q)

print("Expected:", ground_truth[0]["filename"])
print("Text:", text_results[0]["filename"])
print("Vector:", vector_results[0]["filename"])

Expected: 01-agentic-rag/lessons/01-intro.md
Text: 01-agentic-rag/lessons/03-rag.md
Vector: 01-agentic-rag/lessons/01-intro.md


# Q4

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [ ]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)